# Production Forecast & Inventory Decision Pipeline

In [1]:
import pandas as pd
import numpy as np

train = pd.read_pickle("../data/processed/train.pkl")
validation = pd.read_pickle("../data/processed/validation.pkl")
test = pd.read_pickle("../data/processed/test.pkl")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (473436, 15)
Validation: (118664, 15)
Test: (121160, 15)


## Reconstruct Full Historical Demand

In [2]:
import pandas as pd
import numpy as np

# Load raw transaction data
df = pd.read_excel("../data/raw/online_retail.xlsx")

# Remove exact duplicate transactions
df_clean = df.drop_duplicates().copy()

# Identify cancellations
df_clean["is_cancellation"] = df_clean["InvoiceNo"].astype(str).str.startswith("C")

# Identify bad-debt adjustments
df_clean["is_bad_debt"] = df_clean["StockCode"].astype(str).str.upper().eq("B")

# Identify non-product transaction records
non_product_codes = {"D", "M", "POST", "DOT"}

df_clean["is_non_product"] = (
    df_clean["StockCode"].astype(str).str.upper().isin(non_product_codes)
)

# Identify damaged / unsaleable inventory adjustments
df_clean["is_damage"] = (
    df_clean["Description"]
    .fillna("")
    .astype(str)
    .str.contains(r"damaged|damage|unsaleable|destroyed", case=False, regex=True)
)

# Define genuine customer demand
df_clean["is_valid_demand"] = (
    (~df_clean["is_cancellation"])
    & (~df_clean["is_bad_debt"])
    & (~df_clean["is_non_product"])
    & (~df_clean["is_damage"])
    & ~((df_clean["Quantity"] < 0) & (df_clean["UnitPrice"] == 0))
)

# Keep genuine demand
df_demand = df_clean[df_clean["is_valid_demand"]].copy()

# Extract calendar date
df_demand["Date"] = df_demand["InvoiceDate"].dt.normalize()

# Aggregate to SKU × Day
daily_demand = (
    df_demand.groupby(["Date", "StockCode"], as_index=False)["Quantity"]
    .sum()
    .rename(columns={"Quantity": "Demand"})
)

print("Demand rows:", len(daily_demand))
print("Unique SKUs:", daily_demand["StockCode"].nunique())
print("Date range:", daily_demand["Date"].min(), "to", daily_demand["Date"].max())

Demand rows: 276167
Unique SKUs: 3936
Date range: 2010-12-01 00:00:00 to 2011-12-09 00:00:00


## Build Complete SKU Demand Calendar

In [3]:
# Determine each SKU's active period
sku_activity = (
    daily_demand.groupby("StockCode")["Date"]
    .agg(first_date="min", last_date="max")
    .reset_index()
)

# Create a continuous daily calendar for each SKU
sku_calendar = (
    sku_activity.set_index("StockCode")
    .apply(
        lambda row: pd.date_range(row["first_date"], row["last_date"], freq="D"), axis=1
    )
    .explode()
    .reset_index()
    .rename(columns={0: "Date"})
)

# Merge actual demand onto the calendar
demand_daily = sku_calendar.merge(daily_demand, on=["StockCode", "Date"], how="left")

# Missing SKU-days represent zero recorded demand
demand_daily["Demand"] = demand_daily["Demand"].fillna(0).astype(int)

# Sort chronologically
demand_daily = demand_daily.sort_values(["StockCode", "Date"]).reset_index(drop=True)

print("Rows:", len(demand_daily))
print("SKUs:", demand_daily["StockCode"].nunique())
print("Zero-demand rows:", (demand_daily["Demand"] == 0).sum())
print("Positive-demand rows:", (demand_daily["Demand"] > 0).sum())

Rows: 1076751
SKUs: 3936
Zero-demand rows: 800584
Positive-demand rows: 276167


## Save Processed Demand History

In [5]:
demand_daily.to_pickle("../data/processed/demand_daily.pkl")

print("Saved:", demand_daily.shape)

Saved: (1076751, 3)


## Prepare Features for Production Forecasting

In [4]:
# Keep only SKUs eligible for forecasting
sku_activity = (
    demand_daily.groupby("StockCode")["Demand"]
    .agg(active_days=lambda x: (x > 0).sum())
    .reset_index()
)

eligible_skus = sku_activity.loc[sku_activity["active_days"] >= 30, "StockCode"]

production_data = demand_daily[demand_daily["StockCode"].isin(eligible_skus)].copy()

production_data = production_data.sort_values(["StockCode", "Date"]).reset_index(
    drop=True
)

print("Eligible SKUs:", production_data["StockCode"].nunique())
print("Rows:", len(production_data))
print("Date range:", production_data["Date"].min(), "to", production_data["Date"].max())

Eligible SKUs: 2435
Rows: 783875
Date range: 2010-12-01 00:00:00 to 2011-12-09 00:00:00


## Create Forecasting Features

In [5]:
# Lag features
for lag in [1, 7, 14, 28]:
    production_data[f"lag_{lag}"] = production_data.groupby("StockCode")[
        "Demand"
    ].shift(lag)


# Rolling features
for window in [7, 28]:
    production_data[f"rolling_mean_{window}"] = production_data.groupby("StockCode")[
        "Demand"
    ].transform(lambda x: x.shift(1).rolling(window).mean())

production_data["rolling_std_7"] = production_data.groupby("StockCode")[
    "Demand"
].transform(lambda x: x.shift(1).rolling(7).std())


# Calendar features
production_data["day_of_week"] = production_data["Date"].dt.dayofweek

production_data["month"] = production_data["Date"].dt.month

production_data["week_of_year"] = (
    production_data["Date"].dt.isocalendar().week.astype(int)
)

production_data["is_weekend"] = (production_data["day_of_week"] >= 5).astype(int)


# Forecast feature list
model_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "week_of_year",
    "is_weekend",
]

print("Features created:", len(model_features))

Features created: 11


## Train Final Production Model

In [6]:
from xgboost import XGBRegressor

# Keep rows where all forecasting features are available
production_train = production_data.dropna(subset=model_features).copy()

X_production = production_train[model_features]
y_production = production_train["Demand"]

print("Training rows:", len(production_train))
print("Features:", X_production.shape[1])

Training rows: 715695
Features: 11


In [7]:
final_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

final_xgb.fit(X_production, y_production)

print("Final production model trained.")

Final production model trained.


In [8]:
forecast_history = production_data[["StockCode", "Date", "Demand"]].copy()

forecast_history = forecast_history.sort_values(["StockCode", "Date"]).reset_index(
    drop=True
)

forecast_start = forecast_history["Date"].max() + pd.Timedelta(days=1)

forecast_dates = pd.date_range(start=forecast_start, periods=7, freq="D")

print("Forecast start:", forecast_start)
print("Forecast end:", forecast_dates[-1])

Forecast start: 2011-12-10 00:00:00
Forecast end: 2011-12-16 00:00:00


In [11]:
forecast_results = []

for forecast_date in forecast_dates:

    current_rows = []

    for sku in eligible_skus:

        sku_history = forecast_history[
            forecast_history["StockCode"] == sku
        ].sort_values("Date")

        demand_series = sku_history["Demand"]

        # Need at least 28 previous observations
        if len(demand_series) < 28:
            continue

        row = {
            "StockCode": sku,
            "Date": forecast_date,
            "lag_1": demand_series.iloc[-1],
            "lag_7": demand_series.iloc[-7],
            "lag_14": demand_series.iloc[-14],
            "lag_28": demand_series.iloc[-28],
            "rolling_mean_7": demand_series.iloc[-7:].mean(),
            "rolling_std_7": demand_series.iloc[-7:].std(),
            "rolling_mean_28": demand_series.iloc[-28:].mean(),
            "day_of_week": forecast_date.dayofweek,
            "month": forecast_date.month,
            "week_of_year": forecast_date.isocalendar().week,
            "is_weekend": int(forecast_date.dayofweek >= 5),
        }

        current_rows.append(row)

    current_features = pd.DataFrame(current_rows)

    predictions = final_xgb.predict(current_features[model_features])

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    current_features["Predicted_Demand"] = predictions

    forecast_results.append(current_features[["StockCode", "Date", "Predicted_Demand"]])

    # Add today's predictions to history
    new_history = current_features[["StockCode", "Date", "Predicted_Demand"]].rename(
        columns={"Predicted_Demand": "Demand"}
    )

    forecast_history = pd.concat([forecast_history, new_history], ignore_index=True)

    forecast_history = forecast_history.sort_values(["StockCode", "Date"]).reset_index(
        drop=True
    )


future_forecast = pd.concat(forecast_results, ignore_index=True)

print("Forecast rows:", len(future_forecast))
print(
    "Forecast range:",
    future_forecast["Date"].min(),
    "to",
    future_forecast["Date"].max(),
)

Forecast rows: 17045
Forecast range: 2011-12-10 00:00:00 to 2011-12-16 00:00:00


## Optimized Recursive Forecast Generation

In [9]:
# --------------------------------------------------
# Optimized Recursive 7-Day Forecast
# --------------------------------------------------

# Keep latest 28 days for every SKU
forecast_history = (
    production_data[["StockCode", "Date", "Demand"]]
    .sort_values(["StockCode", "Date"])
    .groupby("StockCode")
    .tail(28)
    .copy()
)

forecast_history = forecast_history.sort_values(["StockCode", "Date"]).reset_index(
    drop=True
)


# Forecast dates
forecast_start = demand_daily["Date"].max() + pd.Timedelta(days=1)

forecast_dates = pd.date_range(start=forecast_start, periods=7, freq="D")

forecast_results = []


# --------------------------------------------------
# Recursive forecasting
# --------------------------------------------------

for forecast_date in forecast_dates:

    # Pivot the latest history into rows = SKU
    history_pivot = (
        forecast_history.sort_values(["StockCode", "Date"])
        .groupby("StockCode")["Demand"]
        .apply(list)
    )

    feature_data = pd.DataFrame(
        {
            "StockCode": history_pivot.index,
            "lag_1": history_pivot.apply(lambda x: x[-1]),
            "lag_7": history_pivot.apply(lambda x: x[-7]),
            "lag_14": history_pivot.apply(lambda x: x[-14]),
            "lag_28": history_pivot.apply(lambda x: x[-28]),
            "rolling_mean_7": history_pivot.apply(lambda x: np.mean(x[-7:])),
            "rolling_std_7": history_pivot.apply(lambda x: np.std(x[-7:], ddof=1)),
            "rolling_mean_28": history_pivot.apply(lambda x: np.mean(x[-28:])),
        }
    )

    # Calendar features
    feature_data["day_of_week"] = forecast_date.weekday()

    feature_data["month"] = forecast_date.month

    feature_data["week_of_year"] = forecast_date.isocalendar().week

    feature_data["is_weekend"] = int(forecast_date.weekday() >= 5)

    # Ensure feature order matches training
    predictions = final_xgb.predict(feature_data[model_features])

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    # Store predictions
    current_forecast = feature_data[["StockCode"]].copy()

    current_forecast["Date"] = forecast_date

    current_forecast["Predicted_Demand"] = predictions

    forecast_results.append(current_forecast)

    # Add predictions to history
    new_history = current_forecast[["StockCode", "Date", "Predicted_Demand"]].rename(
        columns={"Predicted_Demand": "Demand"}
    )

    forecast_history = pd.concat([forecast_history, new_history], ignore_index=True)

    # Keep latest 28 observations
    forecast_history = (
        forecast_history.sort_values(["StockCode", "Date"])
        .groupby("StockCode")
        .tail(28)
        .reset_index(drop=True)
    )


# --------------------------------------------------
# Combine forecasts
# --------------------------------------------------

future_forecast = pd.concat(forecast_results, ignore_index=True)


# --------------------------------------------------
# Validation
# --------------------------------------------------

print("Forecast rows:", len(future_forecast))

print(
    "Forecast range:",
    future_forecast["Date"].min(),
    "to",
    future_forecast["Date"].max(),
)

print("Forecast SKUs:", future_forecast["StockCode"].nunique())

Forecast rows: 17045
Forecast range: 2011-12-10 00:00:00 to 2011-12-16 00:00:00
Forecast SKUs: 2435


In [10]:
future_forecast.head(10)

,StockCode,Date,Predicted_Demand
0,10002,2011-12-10,3.028309
1,10125,2011-12-10,1.097516
2,10133,2011-12-10,21.796797
3,10135,2011-12-10,4.103166
4,11001,2011-12-10,5.284906
5,15034,2011-12-10,2.385721
6,15036,2011-12-10,2.552989
7,15039,2011-12-10,2.960327
8,16008,2011-12-10,2.764755
9,16011,2011-12-10,5.054618


In [11]:
future_forecast["Predicted_Demand"].describe()

count    17045.000000
mean        11.105840
std         20.595949
min          0.000000
25%          2.498058
50%          4.800622
75%         12.160699
max        447.685089
Name: Predicted_Demand, dtype: float64

In [12]:
future_forecast.groupby("Date")["Predicted_Demand"].agg(["count", "mean", "sum"])

,count,mean,sum
Date,,,
2011-12-10,2435,2.832361,6896.798828
2011-12-11,2435,5.385560,13113.837891
2011-12-12,2435,14.033536,34171.660156
2011-12-13,2435,14.232757,34656.761719
2011-12-14,2435,14.500795,35309.437500
2011-12-15,2435,14.707601,35813.007812
2011-12-16,2435,12.048266,29337.529297


## Build Forecast-Based Inventory Decisions

In [13]:
# Aggregate 7-day forecast by SKU
sku_forecast = (
    future_forecast.groupby("StockCode")
    .agg(
        forecast_7_day_demand=("Predicted_Demand", "sum"),
        avg_forecast_daily_demand=("Predicted_Demand", "mean"),
    )
    .reset_index()
)

print("SKUs:", sku_forecast["StockCode"].nunique())
print("Rows:", len(sku_forecast))

SKUs: 2435
Rows: 2435


In [14]:
# Historical demand statistics for inventory planning
inventory_stats = (
    production_data.groupby("StockCode")["Demand"]
    .agg(mean_daily_demand="mean", std_daily_demand="std")
    .reset_index()
)

# Inventory planning assumptions
lead_time = 7
service_level = 0.95
z_score = 1.645

# Safety stock
inventory_stats["safety_stock"] = (
    z_score * inventory_stats["std_daily_demand"] * np.sqrt(lead_time)
)

# Expected demand during lead time
inventory_stats["lead_time_demand"] = inventory_stats["mean_daily_demand"] * lead_time

# Historical reorder point
inventory_stats["historical_reorder_point"] = (
    inventory_stats["lead_time_demand"] + inventory_stats["safety_stock"]
)

print("Inventory statistics created.")

Inventory statistics created.


In [15]:
# Combine forecast with inventory statistics
sku_inventory = inventory_stats.merge(sku_forecast, on="StockCode", how="inner")

# Forecast-based reorder point
sku_inventory["forecast_reorder_point"] = (
    sku_inventory["forecast_7_day_demand"] + sku_inventory["safety_stock"]
)

print("Decision rows:", len(sku_inventory))
print("Missing values:", sku_inventory.isna().sum().sum())

Decision rows: 2435
Missing values: 0


In [16]:
sku_inventory[
    ["StockCode", "forecast_7_day_demand", "safety_stock", "forecast_reorder_point"]
].head(10)

,StockCode,forecast_7_day_demand,safety_stock,forecast_reorder_point
0,10002,85.850922,103.259961,189.110883
1,10125,30.053349,61.104338,91.157687
2,10133,169.966507,94.317646,264.284153
3,10135,56.843765,86.659088,143.502853
4,11001,42.011848,95.257843,137.269691
5,15034,38.954163,426.915771,465.869934
6,15036,98.319656,833.824491,932.144148
7,15039,46.532272,106.700674,153.232946
8,16008,206.817291,145.008258,351.825550
9,16011,80.573586,78.135034,158.708619


In [17]:
# Calculate simulated current inventory
recent_inventory = (
    demand_daily[demand_daily["StockCode"].isin(eligible_skus)]
    .sort_values(["StockCode", "Date"])
    .groupby("StockCode")
    .tail(14)
    .groupby("StockCode")["Demand"]
    .sum()
    .reset_index(name="simulated_inventory")
)

print("SKUs:", recent_inventory["StockCode"].nunique())
print("Rows:", len(recent_inventory))
print("Missing inventory values:", recent_inventory["simulated_inventory"].isna().sum())

SKUs: 2435
Rows: 2435
Missing inventory values: 0


In [18]:
sku_inventory = sku_inventory.merge(recent_inventory, on="StockCode", how="inner")

print("Final decision rows:", len(sku_inventory))

Final decision rows: 2435


In [19]:
# Calculate inventory gap
sku_inventory["inventory_gap"] = (
    sku_inventory["forecast_reorder_point"] - sku_inventory["simulated_inventory"]
)

# Percentage gap relative to reorder point
sku_inventory["inventory_gap_pct"] = (
    sku_inventory["inventory_gap"] / sku_inventory["forecast_reorder_point"]
) * 100

# Assign risk levels
sku_inventory["risk_level"] = pd.cut(
    sku_inventory["inventory_gap_pct"],
    bins=[-np.inf, 0, 25, 50, np.inf],
    labels=["Healthy", "Watch", "High", "Critical"],
)

print(
    sku_inventory["risk_level"]
    .value_counts()
    .reindex(["Healthy", "Watch", "High", "Critical"], fill_value=0)
)

risk_level
Healthy     567
Watch       412
High        487
Critical    969
Name: count, dtype: int64


In [20]:
sku_inventory[
    [
        "StockCode",
        "simulated_inventory",
        "forecast_reorder_point",
        "inventory_gap",
        "inventory_gap_pct",
        "risk_level",
    ]
].head(10)

,StockCode,simulated_inventory,forecast_reorder_point,inventory_gap,inventory_gap_pct,risk_level
0,10002,63,189.110883,126.110883,66.686211,Critical
1,10125,26,91.157687,65.157687,71.477995,Critical
2,10133,298,264.284153,-33.715847,-12.757423,Healthy
3,10135,117,143.502853,26.502853,18.468520,Watch
4,11001,76,137.269691,61.269691,44.634537,High
5,15034,18,465.869934,447.869934,96.136261,Critical
6,15036,157,932.144148,775.144148,83.157111,Critical
7,15039,48,153.232946,105.232946,68.675144,Critical
8,16008,624,351.825550,-272.174450,-77.360627,Healthy
9,16011,73,158.708619,85.708619,54.003758,Critical


In [21]:
def recommend_action(risk_level):
    if risk_level == "Critical":
        return "Reorder immediately"
    elif risk_level == "High":
        return "Reorder soon"
    elif risk_level == "Watch":
        return "Monitor closely"
    else:
        return "No action"


sku_inventory["recommended_action"] = sku_inventory["risk_level"].map(recommend_action)

print(
    sku_inventory["recommended_action"]
    .value_counts()
    .reindex(
        ["Reorder immediately", "Reorder soon", "Monitor closely", "No action"],
        fill_value=0,
    )
)

recommended_action
Reorder immediately    969
Reorder soon           487
Monitor closely        412
No action              567
Name: count, dtype: int64


In [22]:
decision_columns = [
    "StockCode",
    "mean_daily_demand",
    "std_daily_demand",
    "safety_stock",
    "forecast_7_day_demand",
    "avg_forecast_daily_demand",
    "forecast_reorder_point",
    "simulated_inventory",
    "inventory_gap",
    "inventory_gap_pct",
    "risk_level",
    "recommended_action",
]

decision_data = sku_inventory[decision_columns].copy()

decision_data.to_csv("../data/processed/stock_inventory_decisions.csv", index=False)

print("Saved:", decision_data.shape)
print("Missing values:", decision_data.isna().sum().sum())

Saved: (2435, 12)
Missing values: 0


In [23]:
decision_data.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock,forecast_7_day_demand,avg_forecast_daily_demand,forecast_reorder_point,simulated_inventory,inventory_gap,inventory_gap_pct,risk_level,recommended_action
0,10002,7.482014,23.725591,103.259961,85.850922,12.264418,189.110883,63,126.110883,66.686211,Critical,Reorder immediately
1,10125,3.462567,14.039677,61.104338,30.053349,4.293335,91.157687,26,65.157687,71.477995,Critical,Reorder immediately
2,10133,10.163701,21.670954,94.317646,169.966507,24.280930,264.284153,298,-33.715847,-12.757423,Healthy,No action
3,10135,5.975871,19.911281,86.659088,56.843765,8.120538,143.502853,117,26.502853,18.468520,Watch,Monitor closely
4,11001,4.353100,21.886979,95.257843,42.011848,6.001693,137.269691,76,61.269691,44.634537,High,Reorder soon


## Save Production Model

In [24]:
import joblib

joblib.dump(final_xgb, "../models/final_xgb_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [25]:
import os

print("Model exists:", os.path.exists("../models/final_xgb_model.pkl"))

Model exists: True


In [26]:
future_forecast.to_csv("../data/processed/future_7_day_forecast.csv", index=False)

print("Forecast saved:", future_forecast.shape)

Forecast saved: (17045, 3)
